In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.13 Optimization: Golden Sections, Amoebas, and Gradient Descent

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Computational Foundations",
    number="0.13",
    title="Optimization: Golden Sections, Amoebas, and Gradient Descent",
    blurb="Finding the bottom: a bracket that shrinks by the golden ratio "
    "and the √ε floor no minimizer can pass, a simplex that crawls "
    "downhill without a single derivative, gradient descent paying the "
    "condition number and momentum paying only its square root, five "
    "charge arrangements on a sphere found by restart, and the "
    "least-squares fits of §0.8 recognized as this subject in disguise.",
    difficulty="introductory",
    estimate="110–140 min",
)

## Notebook overview

Half of computational physics is secretly this notebook. A stable
equilibrium is a minimum of the potential energy; a least-squares fit
([§0.8](fitting-least-squares.ipynb)) is a minimum of $\chi^2$; the
variational method that will carry
[§6.22](../06-quantum-mechanics/variational-method.ipynb) rests on
minimizing an energy expectation over trial wavefunctions. Every one
of those computations ends with the same question — *where is the
bottom?* — and this notebook is where the course answers it
systematically, building the machinery that `scipy.optimize` wraps.

The subject splits along two axes. Dimension one or many: in one
dimension a minimum can be *bracketed* and cornered as surely as
[§0.2](root-finding.ipynb) cornered a root, while in many dimensions
nothing can be fenced in and every method is a strategy for walking
downhill. Derivatives or none: with a gradient in hand, steepest
descent works — at a speed set, we will measure, by the condition
number that [§0.4](linear-systems.ipynb) taught us to fear — and
without one, the Nelder–Mead simplex oozes its way down regardless.
We build both from scratch, certify them against `scipy`, and close
with the honest problem of *global* optimization, where the only
guarantee anyone can sell is a good set of restarts. Nocedal and
Wright {cite}`nocedal2006` and Numerical Recipes {cite}`numrecipes`
are the standing references.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**Bracketing and the golden section.** A minimum is trapped by a
*triplet* $a < m < b$ with $f(m)$ below both ends. To shrink the
bracket we probe one interior point and discard one end — and if we
demand that the two possible surviving brackets have equal width, and
that each round reuse the previous round's interior point, the probe
positions are forced: self-similarity fixes the proportion, and the
proportion is golden,

```{math}
:label: eq-op-golden
w_{k+1} \;=\; (\varphi - 1)\, w_k ,
\qquad
\varphi - 1 = \frac{\sqrt 5 - 1}{2} \approx 0.6180 ,
```

a guaranteed linear convergence with no derivative and no luck
required. But there is a floor. Near the minimum,
$f(x) \approx f(x_*) + \tfrac12 f''(x_*)(x - x_*)^2$, and once the
quadratic term drops below machine precision *of* $f(x_*)$ the
function is computationally flat: no minimizer of function values can
locate $x_*$ better than about $\sqrt{\varepsilon}\,$ (relative), the
square root of the $10^{-16}$ that [§0.1](floating-point.ipynb)
established. Roots can be pinned to $\varepsilon$; minima only to
$\sqrt\varepsilon$. Brent's method (the engine of
`scipy.optimize.minimize_scalar`) reaches that floor faster by fitting
parabolas, with the golden section as its safety net
{cite}`numrecipes`.

**The amoeba.** In $d$ dimensions, Nelder and Mead {cite}`nelder1965`
maintain a *simplex* of $d+1$ points and improve its worst vertex by
trying, in order: reflection through the centroid of the others,
expansion if the reflection excelled, contraction if it disappointed,
and a whole-simplex shrink as the last resort. No gradient, no line
search — just a polytope oozing downhill, which is why everyone calls
it the amoeba. It is the default when derivatives are unavailable or
untrustworthy, and it is `scipy.optimize.minimize(method="Nelder-Mead")`.

**Gradient descent and the condition number.** With a gradient,
$x_{k+1} = x_k - \alpha \nabla f(x_k)$. On the model quadratic
$f = \tfrac12 x^{\!\top}\! A\, x$ with Hessian eigenvalues in
$[\lambda_{\min}, \lambda_{\max}]$ and the optimal fixed step
$\alpha = 2/(\lambda_{\min} + \lambda_{\max})$, the error contracts by
a fixed factor per step,

```{math}
:label: eq-op-gd
\frac{\Vert x_{k+1} - x_* \Vert}{\Vert x_k - x_* \Vert}
\;\longrightarrow\;
\frac{\kappa - 1}{\kappa + 1},
\qquad \kappa = \frac{\lambda_{\max}}{\lambda_{\min}} ,
```

which is the condition number of [§0.4](linear-systems.ipynb) ruling a
third subject: for $\kappa = 100$ each step removes only $2\%$ of the
error, because the step size the steep direction tolerates is far too
timid for the shallow one, and the iterate zig-zags down the valley.
Polyak's remedy {cite}`polyak1964` is *momentum* (the "heavy ball"):
remember the previous displacement,
$x_{k+1} = x_k - \alpha \nabla f(x_k) + \beta\,(x_k - x_{k-1})$, and
with the optimal $\alpha = 4/(\sqrt{\lambda_{\max}} +
\sqrt{\lambda_{\min}})^2$ and $\beta = \bigl((\sqrt\kappa -
1)/(\sqrt\kappa + 1)\bigr)^2$ the rate improves to

```{math}
:label: eq-op-momentum
\frac{\sqrt\kappa - 1}{\sqrt\kappa + 1} ,
```

the condition number's *square root* — the difference between $2\%$
and $18\%$ per step at $\kappa = 100$, and the reason essentially all
large-scale machine learning runs on gradient descent with momentum.

**Local versus global.** Everything above finds the nearest valley.
When a landscape has many — and the energy of $N$ interacting
particles almost always does — the plain, honest tool is *restarts*:
minimize from many random starting points and keep the best. Exercise
4 does exactly that on a problem with a century of history, Thomson's
$N$ electrons on a sphere {cite}`thomson1904`, minimizing

```{math}
:label: eq-op-thomson
E \;=\; \sum_{i<j} \frac{1}{\vert \mathbf r_i - \mathbf r_j \vert},
\qquad \vert \mathbf r_i \vert = 1 ,
```

whose minima are checkable against published values to nine digits.

## Setup

Reduced units throughout (the Lennard-Jones $\varepsilon = \sigma =
1$, unit charges and radii in the Thomson problem). The random
generator is seeded; every restart below is reproducible.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize, minimize_scalar

from ecp import validate

rng = np.random.default_rng(13)

PHI_MINUS_1 = (np.sqrt(5.0) - 1.0) / 2.0  # the golden shrink factor


def lj(r):
    """The Lennard-Jones pair potential 4(r⁻¹² − r⁻⁶) in reduced units.

    Minimum at r = 2^(1/6) ≈ 1.12246 with depth −1: the test bed for
    one-dimensional minimization, chosen because the answer is exact.

    Parameters
    ----------
    r : float or numpy.ndarray
        Pair separation (σ = 1).

    Returns
    -------
    float or numpy.ndarray
        Potential energy (ε = 1).
    """
    return 4.0 * (r**-12 - r**-6)


def golden_section(f, a, b, n_iter):
    """Golden-section search for a minimum of f on [a, b].

    Shrinks the bracket by φ − 1 ≈ 0.618 per iteration, reusing one
    interior evaluation each round (one new f call per iteration).
    Returns the bracket midpoint and the history of bracket widths.

    Parameters
    ----------
    f : callable
        Function to minimize (unimodal on [a, b]).
    a, b : float
        Initial bracket endpoints, a < b.
    n_iter : int
        Number of shrink iterations.

    Returns
    -------
    x_min : float
        Midpoint of the final bracket.
    widths : numpy.ndarray
        Bracket widths after each iteration (length n_iter + 1).
    """
    widths = [b - a]
    x1 = b - PHI_MINUS_1 * (b - a)
    x2 = a + PHI_MINUS_1 * (b - a)
    f1, f2 = f(x1), f(x2)
    for _ in range(n_iter):
        if f1 < f2:
            b, x2, f2 = x2, x1, f1
            x1 = b - PHI_MINUS_1 * (b - a)
            f1 = f(x1)
        else:
            a, x1, f1 = x1, x2, f2
            x2 = a + PHI_MINUS_1 * (b - a)
            f2 = f(x2)
        widths.append(b - a)
    return 0.5 * (a + b), np.array(widths)

## Exercise 1 — The golden section and the √ε floor

The one-dimensional workhorse first, on a function whose minimum is
known exactly: the Lennard-Jones pair potential, with
$r_* = 2^{1/6}$.

**Part a)** Run `golden_section` on `lj` over the bracket $[0.9, 2.0]$
for $80$ iterations. Verify the located minimum agrees with $2^{1/6}$
to `atol=1e-7`, and verify {eq}`eq-op-golden` in action: the mean of
the first $40$ successive width ratios equals $\varphi - 1$ to
`rtol=1e-6`.

**Part b)** The floor. Run `golden_section` again for $200$ iterations —
two and a half times the work — and verify the answer does not move
(the two midpoints agree to $10^{-12}$): by iteration $80$ the bracket
is already below the flatness floor of the theory section, and no
number of further function evaluations can buy another digit. Minima
are $\sqrt\varepsilon$ objects; treat any minimizer's last eight
digits with suspicion.

**Part c)** The professional version: minimize `lj` with
`scipy.optimize.minimize_scalar` (`method="brent"`, bracket
$(0.9, 1.1, 2.0)$) and verify Brent's parabolic acceleration reaches
error below $10^{-6}$ in *fewer than 30* function evaluations — the
golden section's guarantee, at a fraction of its cost.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(x_min80 - R_STAR) < 1e-7,
    "golden-section search pins the Lennard-Jones minimum at 2^(1/6) "
    "to seven digits",
    f"error {abs(x_min80 - R_STAR):.1e}",
)
validate.close(
    np.array([ratios.mean()]),
    np.array([PHI_MINUS_1]),
    "and the bracket shrinks by exactly φ − 1 per iteration: "
    "self-similarity made the proportion golden",
    rtol=1e-6,
)
validate.check(
    abs(x_min80 - x_min200) < 1e-12,
    "120 extra iterations buy NOTHING: the √ε flatness floor is "
    "already reached — minima are half-precision objects",
    f"midpoints differ by {abs(x_min80 - x_min200):.1e}",
)
validate.check(
    abs(res_brent.x - R_STAR) < 1e-6 and res_brent.nfev < 30,
    "Brent's parabolic acceleration reaches the same floor in a "
    "fraction of the evaluations",
    f"error {abs(res_brent.x - R_STAR):.1e} in {res_brent.nfev} calls",
)

## Exercise 2 — The amoeba, from scratch

Now many dimensions without derivatives. The test landscape is
Rosenbrock's banana $f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$ — a
curved, flat-bottomed valley whose minimum $f(1,1) = 0$ is easy to
state and famously tedious to reach.

**Part a)** Implement Nelder–Mead with the four standard moves
(reflect, expand, contract, shrink; coefficients $1, 2, \tfrac12,
\tfrac12$) and run it from $(-1.5, 2.0)$ with initial simplex step
$0.5$ for $300$ iterations. Verify it lands on $(1, 1)$ to
`atol=1e-5` with $f < 10^{-10}$.

**Part b)** Certify against the reference: run
`scipy.optimize.minimize(method="Nelder-Mead")` from the same start
and verify it agrees with $(1, 1)$ to the same tolerance. Plot the
landscape's contours and our simplex's best-vertex path: the amoeba
slides into the valley in a few dozen moves and then feels its way
along the curved floor.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    np.max(np.abs(nm_best - 1.0)) < 1e-5 and nm_fbest < 1e-10,
    "the from-scratch amoeba finds Rosenbrock's minimum (1, 1) with "
    "no derivative information at all",
    f"x = {nm_best}, f = {nm_fbest:.1e}",
)
validate.check(
    np.max(np.abs(res_nm.x - 1.0)) < 1e-5 and res_nm.fun < 1e-10,
    "and scipy's Nelder-Mead, from the same start, certifies the " "answer",
    f"x = {res_nm.x}, f = {res_nm.fun:.1e}",
)
validate.check(
    np.max(np.abs(nm_best - res_nm.x)) < 1e-4,
    "two independent implementations of the same four moves, one " "answer",
    f"gap {np.max(np.abs(nm_best - res_nm.x)):.1e}",
)

## Exercise 3 — Gradient descent pays the condition number

{eq}`eq-op-gd` and {eq}`eq-op-momentum` are quantitative promises,
and a two-dimensional quadratic with Hessian eigenvalues $1$ and
$100$ (so $\kappa = 100$) makes them measurable.

**Part a)** Run gradient descent on $f = \tfrac12(x_1^2 + 100 x_2^2)$
from $(1, 1)$ with the optimal fixed step $\alpha = 2/101$ for $1500$
iterations. Fit (with `numpy.polyfit`) the slope of
$\ln\Vert x_k \Vert$ over iterations $500$–$1400$ and verify the
measured per-step contraction equals $(\kappa - 1)/(\kappa + 1) =
99/101$ to `rtol=1e-6` — theory to six digits.

**Part b)** Add Polyak momentum with the optimal $\alpha = 4/(\sqrt{
\lambda_{\max}} + \sqrt{\lambda_{\min}})^2$ and $\beta = \bigl((\sqrt
\kappa - 1)/(\sqrt\kappa + 1)\bigr)^2$, run $400$ iterations, and
verify the measured rate matches $(\sqrt\kappa - 1)/(\sqrt\kappa + 1)
= 9/11$ to `rtol=1e-2` (the heavy ball oscillates as it converges, so
the fitted rate carries more noise). Verify the practical payoff:
counting iterations to bring $\Vert x_k \Vert$ below $10^{-8}$,
momentum wins by better than a factor of $5$.

**Part c)** The cautionary coda. Run plain gradient descent on
Rosenbrock from $(-1.5, 2.0)$ with step $10^{-3}$ (about the largest
stable choice) for $20{,}000$ iterations and verify the crawl: the
function value falls below $10^{-6}$, yet the iterate is still
farther than $10^{-4}$ from $(1, 1)$ — *twenty thousand* gradient
evaluations against the amoeba's few hundred function calls. On
narrow curved valleys, curvature-blind steps pay {eq}`eq-op-gd`'s
price at every turn; this is the observation that leads to conjugate
gradients and quasi-Newton methods {cite}`nocedal2006`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([rate_gd]),
    np.array([RATE_GD_TH]),
    "gradient descent contracts by (κ−1)/(κ+1) per step, to six "
    "digits: the condition number of §0.4 now rules optimization",
    rtol=1e-6,
)
validate.close(
    np.array([rate_mom]),
    np.array([RATE_MOM_TH]),
    "and the heavy ball contracts by (√κ−1)/(√κ+1): momentum turns "
    "the condition number into its square root",
    rtol=1e-2,
)
validate.check(
    n_gd / n_mom > 5.0,
    "the square root is worth a factor of about eight in iterations " "at κ = 100",
    f"{n_gd} vs {n_mom} iterations to reach 1e-8",
)
validate.check(
    f_crawl < 1e-6 and dist_crawl > 1e-4,
    "on Rosenbrock's curved valley, 20,000 curvature-blind steps "
    "still have not arrived: the case for smarter methods",
    f"f = {f_crawl:.1e} but ‖x − x*‖ = {dist_crawl:.1e}",
)

## Exercise 4 — The Thomson problem: restarts against many valleys

Global optimization, on a problem with real physics pedigree: $N$
unit charges confined to a unit sphere, minimizing
{eq}`eq-op-thomson`. Thomson posed it in 1904 for his model of the
atom {cite}`thomson1904`; its minima for small $N$ are known to many
digits, which makes it the perfect proving ground for the restart
strategy.

**Part a)** Parametrize each charge by spherical angles
$(\theta_i, \phi_i)$, minimize the energy with
`scipy.optimize.minimize` (`method="BFGS"`) from $8$ random starts
for each $N = 2, \dots, 6$, keep the best, and verify all five
ground-state energies against the published values
$0.5$, $\sqrt 3$, $3.674234614$, $6.474691495$, $9.985281374$
to `rtol=1e-8`.

**Part b)** Geometry from optimization: verify the $N = 4$ minimizer
is a regular tetrahedron — all six pairwise distances equal (spread
below $10^{-4}$) with mean $\sqrt{8/3}$ (`rtol=1e-4`) — and plot the
$N = 5$ and $N = 6$ arrangements: the triangular bipyramid and the
octahedron, found by nothing but restarts and descent.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([best_energy[n] for n in range(2, 7)]),
    np.array([THOMSON_REF[n] for n in range(2, 7)]),
    "eight restarts of BFGS recover all five published Thomson ground "
    "states to nine digits: the honest recipe for many valleys",
    rtol=1e-8,
)
validate.check(
    float(np.ptp(dists4)) < 1e-4,
    "the N = 4 minimizer has all six pair distances equal: the "
    "regular tetrahedron, discovered rather than assumed",
    f"spread {np.ptp(dists4):.1e}",
)
validate.close(
    np.array([dists4.mean()]),
    np.array([np.sqrt(8.0 / 3.0)]),
    "with the exact tetrahedral edge length √(8/3) inscribed in the " "unit sphere",
    rtol=1e-4,
)

## Exercise 5 — Closing the circle: fitting is optimization

[§0.8](fitting-least-squares.ipynb) solved least squares by linear
algebra — the normal equations, one shot, no iteration. But the
$\chi^2$ it minimized is just a function of the parameters, and this
notebook minimizes functions.

**Part a)** Generate $30$ noisy samples of $2.5x^2 - 1.2x + 0.7$ on
$[0, 1]$ (Gaussian noise, $\sigma = 0.05$, the seeded generator),
and minimize $\chi^2(\mathbf p) = \sum_i (p(x_i) - y_i)^2$ over
quadratic coefficients with the Exercise 2 machinery
(`scipy.optimize.minimize`, `method="Nelder-Mead"`, tight
tolerances). Verify the amoeba's three coefficients agree with
`numpy.polyfit`'s to `atol=1e-6`, and the two $\chi^2$ values to
`atol=1e-8`: the linear-algebra route and the walking-downhill route
meet at the same bottom, because they are descriptions of the same
minimum.

This is why the distinction matters in practice: when a model is
*linear in its parameters*, [§0.8](fitting-least-squares.ipynb)'s
one-shot algebra is exact and instant — and when it is not (fitting
a decay rate, a resonance width, a phase), there is no algebraic
shortcut, and `scipy.optimize.curve_fit` is quietly running exactly
the kind of iterative descent built in this notebook.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    gap_p < 1e-6,
    "the amoeba walking down the χ² surface lands on numpy.polyfit's "
    "normal-equation answer: fitting IS optimization",
    f"max coefficient gap {gap_p:.1e}",
)
validate.check(
    gap_chi2 < 1e-8,
    "and the two χ² minima are the same number: one bottom, two " "routes to it",
    f"gap {gap_chi2:.1e}",
)

## Notebook summary

- The golden section delivered its guarantee: the Lennard-Jones
  minimum to seven digits, bracket widths shrinking by exactly
  $\varphi - 1$ per step — and $120$ extra iterations bought nothing,
  because minima are $\sqrt\varepsilon$ objects. Brent reached the
  same floor in $16$ evaluations.
- The from-scratch Nelder–Mead amoeba found Rosenbrock's $(1, 1)$
  with no derivatives, matching `scipy`'s implementation of the same
  four moves.
- Gradient descent paid the condition number to six digits —
  contraction $(\kappa - 1)/(\kappa + 1)$ — and Polyak momentum paid
  only its square root, an eight-fold saving at $\kappa = 100$; on
  Rosenbrock's curved valley, $20{,}000$ curvature-blind steps still
  had not arrived.
- Eight random restarts of BFGS recovered all five published Thomson
  ground states to nine digits, and the $N = 4$ minimizer *was* the
  regular tetrahedron ($\sqrt{8/3}$ edges, discovered not assumed).
- The amoeba descending the $\chi^2$ surface landed on
  `numpy.polyfit`'s normal-equation answer exactly: the fits of
  [§0.8](fitting-least-squares.ipynb) were this notebook's subject
  all along.

## Outlook

- **Curvature-aware descent.** The Rosenbrock crawl is cured by
  using second-derivative information: conjugate gradients, Newton,
  and the quasi-Newton BFGS family that Exercise 4 already leaned on
  {cite}`nocedal2006` — which build a Hessian picture from gradient
  history alone.
- **The variational principle.** The course's most consequential
  minimization is quantum: [§6.22](../06-quantum-mechanics/variational-method.ipynb)
  minimizes $\langle\psi_\alpha|H|\psi_\alpha\rangle$ over trial
  wavefunctions, with the guarantee that every value is an upper
  bound on the ground-state energy — this notebook's machinery,
  pointed at the Schrödinger equation.
- **Stochastic escape.** Restarts are one answer to many valleys;
  another is to *accept occasional uphill moves* with a
  temperature-controlled probability and cool slowly — simulated
  annealing, which is the Metropolis engine of
  [§5.8](../05-classical-stat-mech/partition-function.ipynb) run as
  an optimizer.
- **Learning as descent.** Training a neural network is minimizing a
  loss over millions of parameters with noisy gradients — stochastic
  gradient descent with momentum, {eq}`eq-op-momentum` at industrial
  scale. The mathematics of this notebook is, at this writing, the
  mathematics running inside every large model.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()